# 00. Setup, Reproducibility, and the Research Workflow

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook establishes the environment and research habits required before building any neural network.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Select CPU/GPU safely
- Record reproducibility-critical information
- Use explicit experiment configuration
- Design a research folder structure
- Recognize optical-imaging metadata that can confound training


## Mind map

```mermaid
mindmap
  root((Reproducible experiment))
    Environment
      Python
      PyTorch
      CUDA
      Hardware
    Data
      Raw data
      Specimen IDs
      Split manifest
      Metadata
    Experiment
      Config
      Seed
      Checkpoints
      Metrics
    Optical imaging
      Pixel size
      Acquisition settings
      Instrument
      Day/site effects

```


## 1. What reproducibility means in deep learning

A deep-learning result depends on much more than the model architecture. A reproducible experiment records:

| Category | What to record |
|---|---|
| Data | source, specimen IDs, split manifest, preprocessing, patch extraction |
| Code | commit hash, config, model definition, loss, metric implementation |
| Software | Python, PyTorch, CUDA, package versions |
| Hardware | GPU model, memory, number of GPUs |
| Randomness | seed, deterministic settings if required |
| Training | optimizer, learning rate, scheduler, batch size, epochs |
| Model selection | which validation metric selected the checkpoint |
| Evaluation | exact test set and metric definitions |

For optical imaging, also record acquisition conditions that can change image statistics: microscope/system, wavelength, objective, detector, pixel size, exposure/power, reconstruction settings, and staining/contrast conditions when relevant.


In [ ]:
import sys, platform, torch

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime used by PyTorch:", torch.version.cuda)


## 2. CPU, GPU, and device selection

A **device** is where a tensor/model lives. A GPU is useful because neural-network training contains many parallel matrix/tensor operations.

Use one explicit `device` variable so the same code can run on CPU or CUDA. Never assume a GPU exists.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

x = torch.randn(4, 4, device=device)
print(x.device)


## 3. Random seeds: what they do and do not guarantee

A seed helps reproduce pseudo-random operations such as initialization and shuffling. It **does not automatically guarantee bit-for-bit reproducibility** across different hardware, library versions, or nondeterministic kernels.

Use seeds to reduce uncontrolled variation, then report them. For strict reproducibility, consult PyTorch's reproducibility settings and accept that deterministic algorithms can be slower.


In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(torch.randn(3))


## 4. Recommended project structure

```text
project/
├── data/
│   ├── raw/                 # authoritative copies, never overwritten
│   ├── manifests/           # specimen-level train/val/test CSVs
│   └── processed/           # derived data, if needed
├── configs/                 # YAML/JSON experiment parameters
├── notebooks/               # exploration and teaching
├── src/
│   ├── datasets.py
│   ├── models.py
│   ├── losses.py
│   ├── metrics.py
│   └── train.py
├── checkpoints/
├── outputs/
│   ├── figures/
│   └── predictions/
└── results/
    └── metrics.csv
```

**Key rule:** never let the split exist only in your head. Save the actual specimen/file list used for train, validation, and test.


## 5. Configuration instead of hidden constants

Research code becomes hard to audit when important parameters are scattered through many files. Put them in a configuration dictionary/file.


In [ ]:
config = {
    "seed": 42,
    "batch_size": 8,
    "learning_rate": 1e-3,
    "epochs": 50,
    "image_size": 256,
    "normalization": "per_image_percentile",
    "loss": "BCE_plus_Dice",
}

for key, value in config.items():
    print(f"{key:20s}: {value}")


## 6. Optical-imaging reproducibility checklist

Before training, answer:

- What is one **independent biological unit**: patient, animal, embryo, eye, tissue block, slide, or acquisition?
- Are patches from the same unit allowed to appear in different splits? Usually **no**.
- Is pixel size constant across samples?
- Are intensity units quantitative or arbitrary?
- Is preprocessing identical for train/validation/test?
- Could acquisition day, instrument, operator, or site correlate with the labels?
- Does the model see metadata, scale bars, borders, timestamps, or annotations that reveal the answer?
- What raw-data transformations are irreversible?

These questions often matter more than trying a larger network.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
